# RAG biblijny – zadania

## Przygotowanie danych

### Zadanie 1 — Indeksowanie wersetów biblijnych w Qdrant

Wczytaj dane z pliku `data/polskie_przeklady.csv` oraz `data/wszystkie_ksiegi.csv`. Wybierz dowolne tłumaczenie (rekomendowana kolumna to `esp` — Edycja Świętego Pawła) i połącz je z metadanymi ksiąg. Następnie zapisz wersety do wektorowej bazy danych Qdrant w kolekcji `biblia`.

Wskazówki:
- Użyj modelu `sdadas/mmlw-retrieval-roberta-large-v2` z HuggingFace do gęstych embeddingów oraz modelu BM25 (`Qdrant/bm25`) przez `FastEmbedSparse` do wektorów rzadkich — umożliwi to późniejsze wyszukiwanie hybrydowe.
- Każdy werset powinien stanowić osobny punkt w bazie. Jako `id` punktów w kolekcji użyj kolejnych liczb całkowitych
- Payload każdego punktu powinien mieć strukturę:

```json
{
    "page_content": "Na początku Bóg stworzył niebo i ziemię.",
    "metadata": {
        "bible_part": "Stary Testament",
        "book_category": "Pięcioksiąg",
        "book_name": "Księga Rodzaju",
        "book_abbrev": "rdz",
        "deuterocanonical": false,
        "original_language": "hebrajski",
        "chapter": 1,
        "verse": 1
    }
}
```


In [1]:
import numpy as np
import pandas as pd
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import FastEmbedSparse, QdrantVectorStore, RetrievalMode
from qdrant_client import QdrantClient
from qdrant_client.http import models as rest
from qdrant_client.http.models import Distance, VectorParams

2026-05-27 11:45:29.851783195 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card5": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card5/device/vendor"
2026-05-27 11:45:29.851833094 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card3": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card3/device/vendor"
2026-05-27 11:45:29.852057285 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card4": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card4/device/vendor"
2026-05-27 11:45:29.852117189 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card0": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


In [ ]:
df_verses = (
    pd.read_csv("data/polskie_przeklady.csv", usecols=["book_abbrev", "chapter", "verse", "esp"])
    .rename(columns={"esp": "text"})
    .replace(r"^[^\w]*$", np.nan, regex=True)
    .dropna(subset=["text"])
)

df_books = pd.read_csv("data/wszystkie_ksiegi.csv")

df = df_verses.merge(df_books, on="book_abbrev").reset_index(drop=True)
df.head()

In [2]:
embeddings = HuggingFaceEmbeddings(
    model_name="sdadas/mmlw-retrieval-roberta-large-v2",
    # model_kwargs={"device": "cpu"},
)

sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [3]:
COLLECTION_NAME = "biblia"
client = QdrantClient(url="http://localhost:6333")

In [4]:
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
        sparse_vectors_config={
            "langchain-sparse": rest.SparseVectorParams()
        },
    )

vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode=RetrievalMode.HYBRID,
)

/home/patryk/CloudDrive/Szkolenia/!/AI/RAG/.venv/lib/python3.13/site-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.15.5. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


In [5]:
import time

In [ ]:
metadata_cols = ["bible_part", "book_category", "book_name", "book_abbrev", "deuterocanonical", "original_language", "chapter", "verse"]
metadatas = df[metadata_cols].to_dict("records")

start = time.perf_counter()

vector_store.add_texts(
    texts=df["text"].tolist(),
    metadatas=metadatas,
    ids=list(range(len(df))),
)
stop = time.perf_counter()
print(f"Czas: {round(stop - start, 3)}")

## Retrieval i generacja

### Zadanie 2 — Podstawowy RAG: retrieval + generacja odpowiedzi

Napisz funkcję `rag_query`, która dla zadanego pytania wykona retrieval z bazy wektorowej i wygeneruje odpowiedź przy użyciu LLM. Funkcja powinna przyjmować:
- `question` — pytanie użytkownika
- `top_k` — liczba wersetów do pobrania
- `retrieval_mode` — tryb wyszukiwania: `"dense"`, `"sparse"` lub `"hybrid"`

Funkcja powinna zwracać słownik z kluczami `answer` (odpowiedź LLM) oraz `context` (lista słowników, gdzie każdy słownik zawiera tekst wersetu oraz wszystkie pola z payloadu, np. `book_name`, `chapter`, `verse`).

Przetestuj funkcję, zadając różne pytania z różnymi trybami wyszukiwania i wartościami `top_k`.


In [40]:
from dotenv import load_dotenv
from langchain.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors.listwise_rerank import LLMListwiseRerank

load_dotenv()

SYSTEM_PROMPT = (
    "Jesteś asystentem analizy Biblii, który pomaga użytkownikowi poznać stanowisko Biblii w określonych kwestiach. "
    "Dostaniesz w kolejnej wiadomości oryginalne pytanie użytkownika oraz wersety biblijne, które dotyczą tego pytania. "
    "Twoim zadaniem jest odpowiedzieć na pytanie użytkownika w oparciu o przytoczone fragmenty Biblii.\n\n"
    "Zasada krytyczna: odpowiadaj wyłącznie w oparciu o przytoczony kontekst. Nie używaj własnej wiedzy!!! "
    "Nie dokonuj własnej interpretacji!!! Masz być jedynie informatorem przytaczającym naukę Biblii. "
    "POD ŻADNYM POZOREM nie używaj wiedzy spoza dostarczonego tekstu.\n\n"
    "Nie musisz odwoływać się do wszystkich wersetów — użyj tylko tych, które uznasz za najlepiej pasujące do pytania. "
    "Jeżeli wszystkie fragmenty pasują, tylko wtedy możesz odnieść się do wszystkich.\n\n"
    "Twoja odpowiedź powinna być zwięzła i składać się z krótkich, trafnych akapitów przekazujących konkretną myśl. "
    "Każdy akapit powinien zawierać cytowania wersetów popierających tę myśl (jeden lub więcej). "
    "Cała odpowiedź powinna być spójna i prowadzić jedną ciągłą narrację. "
    "Cytowania mogą pochodzić WYŁĄCZNIE z przytoczonych wersetów.\n\n"
    "Cytowania (sigla) podawaj w zapisie katolickim, np. Rdz 1, 2 zamiast Rdz 1:2.\n\n"
    "Odpowiedź powinna zawierać elementy stylowania markdown (pogrubienia, bullet pointy itp.) jeżeli ma to zastosowanie. "
    'Jeśli powołujesz się na dokładny cytat tekstu Biblii, zamknij go w backtikach oraz cudzysłowiach, np. `"Ten tekst jest fragmentem Biblii"`. '
    "Nie używaj dla cytatów pogrubienia ani znaków *. "
    'Cudzysłowy powinny być dokładnie tym znakiem -> " zarówno otwierający jak i zamykający.'
)

HUMAN_PROMPT = "Pytanie użytkownika: {input}\n\nKontekst: {context}"

llm = ChatOpenAI(model="openai/gpt-4o", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    SystemMessage(SYSTEM_PROMPT),
    HumanMessagePromptTemplate.from_template(HUMAN_PROMPT),
])

document_chain = create_stuff_documents_chain(llm, prompt)

In [41]:
def basic_rag_query(question: str, top_k: int, retrieval_mode: str) -> dict:
    mode_map = {
        "dense": RetrievalMode.DENSE,
        "sparse": RetrievalMode.SPARSE,
        "hybrid": RetrievalMode.HYBRID,
    }
    vs = QdrantVectorStore(
        client=client,
        collection_name=COLLECTION_NAME,
        embedding=embeddings,
        sparse_embedding=sparse_embeddings,
        retrieval_mode=mode_map[retrieval_mode],
    )
    retriever = vs.as_retriever(search_kwargs={"k": top_k})
    retrieval_chain = create_retrieval_chain(retriever, document_chain)

    response = retrieval_chain.invoke({"input": question})

    return {
        "answer": response["answer"],
        "context": [
            {"verse_text": doc.page_content} | doc.metadata
            for doc in response["context"]
        ],
    }

In [9]:
result = basic_rag_query("Ile dni trwał biblijny potop?", top_k=5, retrieval_mode="hybrid")
print(result["answer"])

Biblijny potop trwał **czterdzieści dni**. Woda wzbierała i unosiła arkę nad ziemią, co jest podkreślone w kontekście opisu potopu. 

Werset, który to potwierdza, mówi: "Potop trwał czterdzieści dni" (choć nie został podany w dostarczonym kontekście, jest to kluczowy element opisu potopu). 

Dodatkowo, w kontekście dni Noego, można zauważyć, że "I jak było za dni Noego, tak będzie w dniach Syna Człowieczego", co wskazuje na znaczenie tego okresu w historii zbawienia. 

Podsumowując, potop trwał czterdzieści dni, co jest istotnym elementem biblijnej narracji.


In [10]:
print("\n--- Kontekst ---")
for v in result["context"]:
    print(f"{v['book_name']} {v['chapter']},{v['verse']}: {v['verse_text']}")


--- Kontekst ---
Księga Rodzaju 7,17: Potop na ziemi trwał czterdzieści dni. Woda wzbierała i unosiła arkę nad ziemią.
Ewangelia Łukasza 17,26: I jak było za dni Noego, tak będzie w dniach Syna Człowieczego.
Księga Psalmów 119,84: Ile dni zostało Twemu słudze? Kiedy osądzisz moich prześladowców?
Księga Ezechiela 4,4: Potem połóż się na lewym boku i złóż na nim winę ludu izraelskiego. Ile dni będziesz tak leżał, tyle będziesz ponosił karę, na którą oni zasłużyli.
Księga Rodzaju 8,22: Jak długo istnieje ziemia, nie ustanie siew i żniwo, zimno i upał, lato i zima, dzień i noc".


### Zadanie 3 — Reranking pobranych wersetów

Rozszerz funkcję z poprzedniego zadania o opcjonalny reranking. Napisz nową funkcję `rag_query_with_rerank`, która przyjmuje dodatkowy parametr `rerank: bool = False`.

Do implementacji użyj `LLMListwiseRerank` (jako `base_compressor`) owinięty w `ContextualCompressionRetriever`. Gdy `rerank=False`, przekaż do łańcucha zwykły retriever bez kompresji.

Porównaj odpowiedzi i pobrane dokumenty z rerankingiem i bez — zwróć uwagę czy kolejność i dobór wersetów się zmienia.

In [11]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors.listwise_rerank import LLMListwiseRerank

In [42]:
def rag_query_with_rerank(question: str, top_k: int, retrieval_mode: str, rerank: bool = False) -> dict:
    mode_map = {
        "dense": RetrievalMode.DENSE,
        "sparse": RetrievalMode.SPARSE,
        "hybrid": RetrievalMode.HYBRID,
    }
    vs = QdrantVectorStore(
        client=client,
        collection_name=COLLECTION_NAME,
        embedding=embeddings,
        sparse_embedding=sparse_embeddings,
        retrieval_mode=mode_map[retrieval_mode],
    )
    base_retriever = vs.as_retriever(search_kwargs={"k": top_k})

    if rerank:
        reranker = LLMListwiseRerank.from_llm(llm=llm, top_n=top_k)
        retriever = ContextualCompressionRetriever(
            base_compressor=reranker,
            base_retriever=base_retriever,
        )
    else:
        retriever = base_retriever

    retrieval_chain = create_retrieval_chain(retriever, document_chain)
    response = retrieval_chain.invoke({"input": question})

    return {
        "answer": response["answer"],
        "context": [
            {"verse_text": doc.page_content} | doc.metadata
            for doc in response["context"]
        ],
    }


result = rag_query_with_rerank("Ile dni trwał biblijny potop?", top_k=5, retrieval_mode="hybrid", rerank=True)
print(result["answer"])
print("\n--- Kontekst ---")
for v in result["context"]:
    print(f"{v['book_name']} {v['chapter']},{v['verse']}: {v['verse_text']}")

Na podstawie przytoczonego kontekstu, biblijny potop trwał czterdzieści dni. Woda wzbierała i unosiła arkę nad ziemią przez ten czas. Fragment, który to potwierdza, brzmi: `"Potop na ziemi trwał czterdzieści dni. Woda wzbierała i unosiła arkę nad ziemią."`

--- Kontekst ---
Księga Rodzaju 7,17: Potop na ziemi trwał czterdzieści dni. Woda wzbierała i unosiła arkę nad ziemią.
Ewangelia Łukasza 17,26: I jak było za dni Noego, tak będzie w dniach Syna Człowieczego.
Księga Rodzaju 8,22: Jak długo istnieje ziemia, nie ustanie siew i żniwo, zimno i upał, lato i zima, dzień i noc".
Księga Ezechiela 4,4: Potem połóż się na lewym boku i złóż na nim winę ludu izraelskiego. Ile dni będziesz tak leżał, tyle będziesz ponosił karę, na którą oni zasłużyli.
Księga Psalmów 119,84: Ile dni zostało Twemu słudze? Kiedy osądzisz moich prześladowców?


### Zadanie 4 — Filtrowanie payloadu i opcjonalna generacja

Napisz funkcję `rag_query_with_filters`, która rozszerza poprzednie rozwiązanie o dwie nowe możliwości:

**Filtrowanie payloadu** — funkcja powinna przyjmować opcjonalny parametr `filters` z obiektem `models.Filter` z biblioteki `qdrant_client`. Filtry pozwalają zawęzić wyszukiwanie wyłącznie do wersetów spełniających określone warunki metadanych (np. tylko Stary Testament, bez ksiąg deuterokanonicznych, tylko konkretna kategoria ksiąg). Ponieważ filtry muszą być przekazane dynamicznie w momencie wywołania, użyj `configurable_fields` na retrieverze wraz z `ConfigurableField` — dzięki temu `search_kwargs` (w tym `filter`) można podać w `config` przy każdym wywołaniu zamiast przy tworzeniu retrievera.

**Opcjonalne generowanie odpowiedzi** — dodaj parametr `generate_answer: bool = True`. Gdy `True`, uruchom pełny łańcuch RAG i zwróć odpowiedź LLM. Gdy `False`, wywołaj retriever bezpośrednio z pytaniem i zwróć tylko pobrany kontekst bez angażowania LLM.

Funkcja powinna zwracać słownik z `answer` (lub `None` gdy `generate_answer=False`) oraz `context` z pełnymi metadanymi każdego wersetu.

Napisz też pomocniczą funkcję `show_context`, która w czytelny sposób wydrukuje listę wersetów z kontekstu (m.in. część Biblii, kategorię księgi, nazwę, rozdział i werset).

Przetestuj różne typy filtrów: `MatchValue`, `MatchAny`, `MatchText`, kombinacje `must`/`should`/`must_not` oraz reranking z filtrami.


In [18]:
from operator import itemgetter
from langchain_core.runnables import ConfigurableField
from qdrant_client import models


def rag_query_with_filters(
    question: str,
    top_k: int,
    retrieval_mode: str,
    filters=None,
    rerank: bool = False,
    generate_answer: bool = True,
) -> dict:
    mode_map = {
        "dense": RetrievalMode.DENSE,
        "sparse": RetrievalMode.SPARSE,
        "hybrid": RetrievalMode.HYBRID,
    }
    vs = QdrantVectorStore(
        client=client,
        collection_name=COLLECTION_NAME,
        embedding=embeddings,
        sparse_embedding=sparse_embeddings,
        retrieval_mode=mode_map[retrieval_mode],
    )
    base_retriever = vs.as_retriever().configurable_fields(
        search_kwargs=ConfigurableField(id="search_kwargs")
    )

    if rerank:
        reranker = LLMListwiseRerank.from_llm(llm=llm, top_n=top_k)
        base_retriever = ContextualCompressionRetriever(
            base_compressor=reranker,
            base_retriever=base_retriever,
        )

    search_kwargs = {"k": top_k}
    if filters is not None:
        search_kwargs["filter"] = filters

    invoke_config = {"configurable": {"search_kwargs": search_kwargs}}

    if generate_answer:
        retriever = itemgetter("input") | base_retriever
        retrieval_chain = create_retrieval_chain(retriever, document_chain)
        response = retrieval_chain.invoke({"input": question}, config=invoke_config)
        answer = response["answer"]
        docs = response["context"]
    else:
        docs = base_retriever.invoke(question, config=invoke_config)
        answer = None

    return {
        "answer": answer,
        "context": [
            {"verse_text": doc.page_content} | doc.metadata
            for doc in docs
        ],
    }


def show_context(result):
    for v in result["context"]:
        print(f"  [{v.get('bible_part', '')} | {v.get('book_category', '')}] "
              f"{v['book_name']} {v['chapter']},{v['verse']}: {v['verse_text']}")

In [21]:
# --- 1. must + FieldCondition + MatchValue ---
# Tylko Stary Testament
print("=== must: tylko Stary Testament ===")
result = rag_query_with_filters(
    "Co Bóg nakazał ludziom?", top_k=4, retrieval_mode="hybrid",
    filters=models.Filter(
        must=[models.FieldCondition(key="metadata.bible_part", match=models.MatchValue(value="Stary Testament"))]
    ),
    generate_answer=False,
)
show_context(result)

=== must: tylko Stary Testament ===
  [Stary Testament | Pięcioksiąg] Księga Powtórzonego Prawa 5,32: Starajcie się więc wypełnić wszystko, co wam nakazał PAN, wasz Bóg. Nie odstępujcie od tego ani na krok.
  [Stary Testament | Księgi prorockie] Księga Izajasza 27,7: Czyż Bóg chłostał go tak, jak smagał jego oprawców? Czy go zabił, jak zabił jego zabójców?
  [Stary Testament | Księgi mądrościowe] Księga Hioba 37,15: Czy wiesz, jakie Bóg zleca im zadania? W jaki sposób zapala światło w obłokach?
  [Stary Testament | Księgi mądrościowe] Księga Koheleta 1,13: I postanowiłem sobie w duchu rozważać i zgłębiać z pomocą mądrości to wszystko, co się dokonuje pod słońcem. Jest to uciążliwe zajęcie, które Bóg dał ludziom, by się nim trudzili.


In [25]:
# --- 2. must_not + FieldCondition + MatchValue ---
# Wyklucz księgi deuterokanoniczne
print("=== must_not: bez ksiąg deuterokanonicznych ===")
result = rag_query_with_filters(
    "Kim byli Machabeusze?", top_k=4, retrieval_mode="hybrid",
    filters=models.Filter(
        must_not=[models.FieldCondition(key="metadata.deuterocanonical", match=models.MatchValue(value=True))]
    ),
    generate_answer=False,
)
show_context(result)

=== must_not: bez ksiąg deuterokanonicznych ===
  [Stary Testament | Pięcioksiąg] Księga Rodzaju 46,23: Synem Dana był Chuszim.
  [Nowy Testament | Listy apostolskie] 2 List do Tymoteusza 3,14: Ty natomiast trwaj w tym, czego się nauczyłeś i co ci powierzono, wiedząc, kim byli twoi nauczyciele!
  [Stary Testament | Księgi historyczne] Księga Estery 7,1: Król i Haman udali się na ucztę do królowej.
  [Nowy Testament | Ewangelie] Ewangelia Łukasza 9,18: Kiedy pewnego razu Jezus modlił się na osobności, a uczniowie byli razem z Nim, zapytał ich: "Co mówią tłumy? Kim według nich jestem?.


In [28]:
# --- 3. should + FieldCondition + MatchValue ---
# Język grecki LUB Księga Rodzaju
print("=== should: grecki lub aramejski ===")
result = rag_query_with_filters(
    "Opisz stworzenie świata", top_k=4, retrieval_mode="dense",
    filters=models.Filter(
        should=[
            models.FieldCondition(key="metadata.original_language", match=models.MatchValue(value="grecki")),
            models.FieldCondition(key="metadata.book_name", match=models.MatchValue(value="Księga Rodzaju")),
        ]
    ),
    generate_answer=False,
)
show_context(result)

=== should: grecki lub aramejski ===
  [Stary Testament | Pięcioksiąg] Księga Rodzaju 1,1: Na początku Bóg stworzył niebo i ziemię.
  [Stary Testament | Pięcioksiąg] Księga Rodzaju 2,4: Takie są dzieje stworzenia nieba i ziemi.
  [Stary Testament | Księgi mądrościowe] Mądrość Syracha 18,1: Żyjący na wieki stworzył wszystko.  
  [Nowy Testament | Listy apostolskie] List do Kolosan 3,2: Rozmyślajcie o tym, co przewyższa rzeczy ziemskie,


In [29]:
# --- 4. MatchAny (isin – zbiór wartości) ---
# Kategoria księgi należy do zbioru
print("=== MatchAny (isin): Pięcioksiąg lub Ewangelie ===")
result = rag_query_with_filters(
    "Jak należy się modlić?", top_k=4, retrieval_mode="hybrid",
    filters=models.Filter(
        must=[models.FieldCondition(
            key="metadata.book_category",
            match=models.MatchAny(any=["Pięcioksiąg", "Ewangelie"]),
        )]
    ),
    generate_answer=False,
)
show_context(result)

=== MatchAny (isin): Pięcioksiąg lub Ewangelie ===
  [Nowy Testament | Ewangelie] Ewangelia Łukasza 11,1: Jezus modlił się w pewnym miejscu. Kiedy skończył, jeden z Jego uczniów poprosił Go: "Panie, naucz nas modlić się, jak Jan nauczył swoich uczniów.
  [Nowy Testament | Ewangelie] Ewangelia Mateusza 24,20: Módlcie się, abyście nie musieli uciekać w zimie lub w szabat.
  [Stary Testament | Pięcioksiąg] Księga Wyjścia 12,26: Wówczas wasi synowie będą was pytać: «Czym dla was jest ten obrzęd?».
  [Nowy Testament | Ewangelie] Ewangelia Marka 10,14: Kiedy Jezus to zobaczył, oburzył się i powiedział: "Pozwólcie dzieciom przychodzić do Mnie, nie przeszkadzajcie im, bo do takich jak one należy królestwo Boże.


In [30]:
# --- 5. MatchText (contains – zawiera podciąg) ---
# Nazwa księgi zawiera "Królewska"
print("=== MatchText (contains): book_name zawiera 'Królewska' ===")
result = rag_query_with_filters(
    "Kto był królem Izraela?", top_k=4, retrieval_mode="hybrid",
    filters=models.Filter(
        must=[models.FieldCondition(key="metadata.book_name", match=models.MatchText(text="Królewska"))]
    ),
    generate_answer=False,
)
show_context(result)

=== MatchText (contains): book_name zawiera 'Królewska' ===
  [Stary Testament | Księgi historyczne] 1 Księga Królewska 4,1: Król Salomon był władcą całego Izraela.
  [Stary Testament | Księgi historyczne] 1 Księga Królewska 11,25: Był on przeciwnikiem Izraela przez cały okres życia Salomona i podobnie jak Hadad, gdy został królem Aramu, znienawidził Izraela.
  [Stary Testament | Księgi historyczne] 1 Księga Królewska 22,2: Kiedy w trzecim roku król Judy Jozafat przybył do króla izraelskiego,
  [Stary Testament | Księgi historyczne] 2 Księga Królewska 14,26: PAN bowiem widział, jak gorzki los stał się udziałem Izraela: każdy bez wyjątku był uciśniony i opuszczony. Nie było nikogo, kto by wspomógł Izraela.


In [31]:
# --- 6. must + must_not + rerank=True ---
# Nowy Testament bez Listów apostolskich, z rerankingiem
print("=== must + must_not + rerank: Nowy Testament bez Listów apostolskich ===")
result = rag_query_with_filters(
    "Co Jezus mówił o miłości?", top_k=4, retrieval_mode="hybrid",
    filters=models.Filter(
        must=[models.FieldCondition(key="metadata.bible_part", match=models.MatchValue(value="Nowy Testament"))],
        must_not=[models.FieldCondition(key="metadata.book_category", match=models.MatchValue(value="Listy apostolskie"))],
    ),
    rerank=True,
    generate_answer=False,
)
show_context(result)

=== must + must_not + rerank: Nowy Testament bez Listów apostolskich ===
  [Nowy Testament | Listy apostolskie] 2 list do Koryntian 11,11: Dlaczego? Czy dlatego, że was nie miłuję? Bóg to wie.
  [Nowy Testament | Ewangelie] Ewangelia Jana 6,61: Jezus świadom, że Jego uczniowie oburzają się na to, co mówił, zwrócił się do nich: "Tak was to gorszy?
  [Nowy Testament | Ewangelie] Ewangelia Łukasza 23,34: Jezus mówił: "Ojcze, odpuść im, bo nie wiedzą, co czynią". Oni zaś rzucili losy o Jego ubranie i podzielili się nim.
  [Nowy Testament | Listy apostolskie] 1 List do Tesaloniczan 4,2: Znacie przecież polecenia, jakie daliśmy wam w imię Pana Jezusa.


## Asystent obsługujący retrieval

### Zadanie 5 — Asystent generujący zapytania do retrievera

Stwórz funkcję `generate_retrieval_requests`, która pełni rolę inteligentnego asystenta przygotowującego zapytania do bazy wektorowej. Funkcja przyjmuje:
- `question` – oryginalne pytanie użytkownika
- `analysis_level` – poziom analizy: `"basic"`, `"medium"` lub `"deep"`

Asystent powinien wygenerować 2–3 zapytania (lub jedno, jeśli pytanie jest bardzo specyficzne), z których każde pokrywa inny aspekt tematu — dzięki temu retrieval znajdzie szerszy zestaw trafnych wersetów. 

Przykład:
- Pytanie użytkownika: "Nakazy od Boga względem człowieka"
- Wygenerowane zapytanie 1: `query_text` - "Przykazania od Boga wobec Izraelitów", `query_filters` - `<tylko stary testament>`, `top_k` - 20
- Wygenerowane zapytanie 2: `query_text` - "Przykazania Jezusa wobec jego wyznawców", `query_filters` - `<tylko nowy testament>`, `top_k` - 20

Każde zapytanie to słownik z kluczami:
- `query_text` — przeformułowane pytanie zoptymalizowane pod wyszukiwanie semantyczne (bez nazw ksiąg/części Biblii, które trafią do filtrów)
- `query_filters` — gotowy obiekt `models.Filter` z `qdrant_client` lub `None`, gdy filtrowanie nie jest potrzebne; filtry stosuj **wyłącznie** gdy użytkownik wprost wskazuje konkretną część Biblii
- `top_k` — liczba wyników dobrana do poziomu analizy i popularności tematu

Do wygenerowania zapytań użyj LLM ze structured output (Pydantic + `with_structured_output`). Zdefiniuj modele Pydantic opisujące strukturę pojedynczego zapytania i całej listy.

Aby asystent wiedział jakie wartości są prawidłowe dla filtrów, wczytaj plik `data/unique_metadata_values.json` i użyj zawartych w nim informacji.

Ustal samodzielnie mapowanie pomiędzy `analysis_level` → zakres `top_k` . Im bardziej dogłębna analiza, tym więcej wersetów nalezy wyciągnąć.

In [43]:
import json
from pydantic import BaseModel, Field
from qdrant_client import models as qdrant_models

with open("data/unique_metadata_values.json") as _f:
    _VALID_METADATA = json.load(_f)

METADATA_JSON = json.dumps(_VALID_METADATA, ensure_ascii=False)
METADATA_JSON

'{"bible_part": ["Stary Testament", "Nowy Testament"], "book_category": ["Księgi mądrościowe", "Księgi prorockie", "Księgi historyczne", "Apokalipsa", "Pięcioksiąg", "Dzieje Apostolskie", "Listy apostolskie", "Ewangelie"], "book_name": ["Księga Rodzaju", "Księga Wyjścia", "Księga Kapłańska", "Księga Liczb", "Księga Powtórzonego Prawa", "Księga Jozuego", "Księga Sędziów", "Księga Rut", "1 Księga Samuela", "2 Księga Samuela", "1 Księga Królewska", "2 Księga Królewska", "1 Księga Kronik", "2 Księga Kronik", "Księga Ezdrasza", "Księga Nehemiasza", "Księga Estery", "Księga Hioba", "Księga Psalmów", "Księga Przysłów", "Księga Koheleta", "Pieśń nad pieśniami", "Księga Izajasza", "Księga Jeremiasza", "Lamentacje Jeremiasza", "Księga Ezechiela", "Księga Daniela", "Księga Ozeasza", "Księga Joela", "Księga Amosa", "Księga Abdiasza", "Księga Jonasza", "Księga Micheasza", "Księga Nahuma", "Księga Habakuka", "Księga Sofoniasza", "Księga Aggeusza", "Księga Zachariasza", "Księga Malachiasza", "Księga 

In [44]:
def get_top_k_range(analysis_level: str) -> tuple[int, int, int]:
    """Returns (popular_topic_top_k, standard_topic_top_k, specific_topic_top_k)."""
    if analysis_level == "basic":
        return 5, 3, 2
    elif analysis_level == "medium":
        return 15, 10, 5
    elif analysis_level == "deep":
        return 30, 20, 10
    raise ValueError(f"Unknown analysis_level: {analysis_level!r}")


class QdrantCondition(BaseModel):
    key: str = Field(description=(
        "Klucz pola w payloadzie, np. 'metadata.bible_part'. "
        "Zawsze poprzedź nazwę pola prefiksem 'metadata.'."
    ))
    value: str = Field(description=(
        "Wartość pola – MUSI być dosłownie jedną z wartości z listy unikalnych wartości metadanych. "
        "Skopiuj wartość znak w znak."
    ))


class QdrantFilter(BaseModel):
    must: list[QdrantCondition] = Field(
        default_factory=list,
        description="Warunki AND – WSZYSTKIE muszą być spełnione jednocześnie.",
    )
    should: list[QdrantCondition] = Field(
        default_factory=list,
        description="Warunki OR – przynajmniej jeden musi być spełniony.",
    )
    must_not: list[QdrantCondition] = Field(
        default_factory=list,
        description="Warunki NOT – żaden nie może być spełniony.",
    )


def build_request_model(top_k_popular: int, top_k_standard: int, top_k_specific: int):
    class RetrievalRequest(BaseModel):
        query_text: str = Field(description=(
            "Zapytanie zoptymalizowane pod kątem semantycznego wyszukiwania wersetów Biblii. "
            "Przeformułuj pytanie użytkownika tak, aby retriever odnalazł jak najlepiej pasujące wersety. "
            "Nie zawieraj w nim informacji o konkretnej księdze lub części Biblii – te trafią do filtrów. "
            "Nie używaj słowa 'Biblia' – skup się wyłącznie na samym temacie."
        ))
        query_filters: QdrantFilter = Field(
            default_factory=QdrantFilter,
            description=(
                "Filtry Qdrant – używaj WYŁĄCZNIE gdy użytkownik wprost wskazuje konkretną część lub "
                f"księgę Biblii. Dostępne pola i ich wartości: {METADATA_JSON}. "
                "Jeśli filtrowanie nie jest potrzebne, pozostaw wszystkie listy puste."
            ),
        )
        top_k: int = Field(
            ge=top_k_specific,
            le=top_k_popular,
            description=(
                "Liczba wersetów do pobrania. Dobierz na podstawie popularności tematu: "
                f"temat popularny (wiele pasujących fragmentów): {top_k_popular}, "
                f"temat standardowy: {top_k_standard}, "
                f"temat bardzo specyficzny (jedno konkretne miejsce w Biblii): {top_k_specific}."
            ),
        )
        reasoning: str = Field(
            description="Krótkie uzasadnienie wyborów query_text, query_filters i top_k."
        )

    class RetrievalRequests(BaseModel):
        requests: list[RetrievalRequest] = Field(
            description=(
                "Lista 2-3 zapytań do retrievera pokrywających różne aspekty pytania użytkownika. "
                "Każde zapytanie musi dotyczyć innego aspektu – nie duplikuj tematu między requestami. "
                "Jeśli pytanie jest proste i specyficzne, jedno zapytanie wystarczy."
            )
        )

    return RetrievalRequests


def to_qdrant_filter(qf: QdrantFilter) -> qdrant_models.Filter | None:
    if not (qf.must or qf.should or qf.must_not):
        return None

    def to_cond(c: QdrantCondition):
        return qdrant_models.FieldCondition(key=c.key, match=qdrant_models.MatchValue(value=c.value))

    return qdrant_models.Filter(
        must=[to_cond(c) for c in qf.must] or None,
        should=[to_cond(c) for c in qf.should] or None,
        must_not=[to_cond(c) for c in qf.must_not] or None,
    )


SYSTEM_PROMPT_RETRIEVAL_ASSISTANT = (
    "Jesteś biblistą zajmującym się wyszukiwaniem wersetów biblijnych za pomocą retrievera wektorowego. "
    "Analizujesz pytanie użytkownika i generujesz zestaw zapytań do systemu wyszukiwania semantycznego. "
    "Każde zapytanie powinno pokrywać inny aspekt pytania, aby zmaksymalizować pokrycie tematyczne "
    "i liczbę odnalezionych pasujących wersetów."
)

HUMAN_PROMPT_RETRIEVAL_ASSISTANT = "Pytanie użytkownika: {question}"

retrieval_assistant_prompt = ChatPromptTemplate.from_messages([
    SystemMessage(SYSTEM_PROMPT_RETRIEVAL_ASSISTANT),
    HumanMessagePromptTemplate.from_template(HUMAN_PROMPT_RETRIEVAL_ASSISTANT),
])


def generate_retrieval_requests(question: str, analysis_level: str) -> list[dict]:
    top_k_popular, top_k_standard, top_k_specific = get_top_k_range(analysis_level)
    model_cls = build_request_model(top_k_popular, top_k_standard, top_k_specific)

    structured_llm = llm.with_structured_output(model_cls)  #, method="function_calling")
    chain = retrieval_assistant_prompt | structured_llm

    result = chain.invoke({"question": question})

    return [
        {
            "query_text": req.query_text,
            "query_filters": to_qdrant_filter(req.query_filters),
            "top_k": req.top_k,
        }
        for req in result.requests
    ]

In [45]:
# Przykład 1: pytanie ogólne bez filtrów → wiele aspektów, poziom medium
requests = generate_retrieval_requests(
    "Jaka relacja łączy człowieka z Bogiem w starym a jaka w nowym testamencie?",
    analysis_level="medium",
)

for i, req in enumerate(requests, 1):
    print(f"=== Zapytanie {i} ===")
    print(f"  query_text:    {req['query_text']}")
    print(f"  query_filters: {req['query_filters']}")
    print(f"  top_k:         {req['top_k']}")
    print()

=== Zapytanie 1 ===
  query_text:    Relacja człowieka z Bogiem w Starym Testamencie
  query_filters: should=None min_should=None must=[FieldCondition(key='metadata.bible_part', match=MatchValue(value='Stary Testament'), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None)] must_not=None
  top_k:         10

=== Zapytanie 2 ===
  query_text:    Relacja człowieka z Bogiem w Nowym Testamencie
  query_filters: should=None min_should=None must=[FieldCondition(key='metadata.bible_part', match=MatchValue(value='Nowy Testament'), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None)] must_not=None
  top_k:         10

=== Zapytanie 3 ===
  query_text:    Zmiany w relacji człowieka z Bogiem między Starym a Nowym Testamentem
  query_filters: None
  top_k:         15



In [47]:
# Przykład 2: pytanie z ograniczeniem do konkretnej części Biblii → filtry powinny się pojawić
requests = generate_retrieval_requests(
    "Co Nowy Testament mówi o zbawieniu przez wiarę?",
    analysis_level="basic",
)

for i, req in enumerate(requests, 1):
    print(f"=== Zapytanie {i} ===")
    print(f"  query_text:    {req['query_text']}")
    print(f"  query_filters: {req['query_filters']}")
    print(f"  top_k:         {req['top_k']}")
    print()

=== Zapytanie 1 ===
  query_text:    zbawienie przez wiarę w Jezusa Chrystusa
  query_filters: should=None min_should=None must=[FieldCondition(key='metadata.bible_part', match=MatchValue(value='Nowy Testament'), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None)] must_not=None
  top_k:         3

=== Zapytanie 2 ===
  query_text:    rola wiary w procesie zbawienia
  query_filters: should=None min_should=None must=[FieldCondition(key='metadata.bible_part', match=MatchValue(value='Nowy Testament'), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None)] must_not=None
  top_k:         3

=== Zapytanie 3 ===
  query_text:    nauki Jezusa o zbawieniu przez wiarę
  query_filters: should=None min_should=None must=[FieldCondition(key='metadata.bible_part', match=MatchValue(value='Nowy Testament'), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None

In [51]:
requests = generate_retrieval_requests(
    "Ile dni trwał biblijny potop?",
    analysis_level="deep",
)
requests

[{'query_text': 'Czas trwania potopu opisanego w historii Noego',
  'query_filters': None,
  'top_k': 10}]

In [50]:
print("Wygenerowane zapytania:")
for i, req in enumerate(requests, 1):
    print(f"  {i}. [{req['top_k']} wyników] {req['query_text']}")
    if req["query_filters"]:
        print(f"     filtry: {req['query_filters']}")

print()
print("--- Retrieval z wygenerowanymi zapytaniami ---")
all_docs = []
for req in requests:
    docs = rag_query_with_filters(
        question=req["query_text"],
        top_k=req["top_k"],
        retrieval_mode="hybrid",
        filters=req["query_filters"],
        generate_answer=False,
    )
    all_docs.extend(docs["context"])

# Deduplikacja po (book_name, chapter, verse)
seen = set()
unique_docs = []
for doc in all_docs:
    key = (doc["book_name"], doc["chapter"], doc["verse"])
    if key not in seen:
        seen.add(key)
        unique_docs.append(doc)

print(f"Znaleziono {len(unique_docs)} unikalnych wersetów:")
for v in unique_docs:
    print(f"  [{v.get('bible_part', '')} | {v.get('book_category', '')}] "
          f"{v['book_name']} {v['chapter']},{v['verse']}: {v['verse_text']}")

Wygenerowane zapytania:
  1. [10 wyników] Czas trwania potopu opisanego w historii Noego

--- Retrieval z wygenerowanymi zapytaniami ---
Znaleziono 10 unikalnych wersetów:
  [Stary Testament | Księgi mądrościowe] Księga Psalmów 81,4: Dmijcie w trąbę w czas nowiu i w czas pełni, w nasze święta.
  [Stary Testament | Pięcioksiąg] Księga Rodzaju 7,17: Potop na ziemi trwał czterdzieści dni. Woda wzbierała i unosiła arkę nad ziemią.
  [Nowy Testament | Ewangelie] Ewangelia Łukasza 17,26: I jak było za dni Noego, tak będzie w dniach Syna Człowieczego.
  [Stary Testament | Księgi historyczne] Księga Tobiasza 4,9: W ten sposób zgromadzisz sobie w niebie wspaniały skarb na czas niepomyślny 
  [Stary Testament | Pięcioksiąg] Księga Rodzaju 7,6: Miał on sześćset lat, gdy przyszedł potop na ziemię.
  [Nowy Testament | Dzieje Apostolskie] Dzieje Apostolskie 9,43: Piotr zaś jeszcze przez dłuższy czas pozostał w Jafie w domu garbarza Szymona.
  [Stary Testament | Pięcioksiąg] Księga Rodzaju 7,10: Po u

In [52]:
all_docs

[{'verse_text': 'Dmijcie w trąbę w czas nowiu i w czas pełni, w nasze święta.',
  'bible_part': 'Stary Testament',
  'book_category': 'Księgi mądrościowe',
  'book_name': 'Księga Psalmów',
  'book_abbrev': 'ps',
  'deuterocanonical': False,
  'original_language': 'hebrajski',
  'chapter': 81,
  'verse': 4,
  '_id': 15217,
  '_collection_name': 'biblia'},
 {'verse_text': 'Potop na ziemi trwał czterdzieści dni. Woda wzbierała i unosiła arkę nad ziemią.',
  'bible_part': 'Stary Testament',
  'book_category': 'Pięcioksiąg',
  'book_name': 'Księga Rodzaju',
  'book_abbrev': 'rdz',
  'deuterocanonical': False,
  'original_language': 'hebrajski',
  'chapter': 7,
  'verse': 17,
  '_id': 176,
  '_collection_name': 'biblia'},
 {'verse_text': 'I jak było za dni Noego, tak będzie w dniach Syna Człowieczego.',
  'bible_part': 'Nowy Testament',
  'book_category': 'Ewangelie',
  'book_name': 'Ewangelia Łukasza',
  'book_abbrev': 'luk',
  'deuterocanonical': False,
  'original_language': 'grecki',
  '

### Zadanie 6 — Rozszerzanie kontekstu wersetu o sąsiednie wersety

Napisz funkcję `expand_verse_context`, która dla podanego wersetu pobiera z bazy danych sąsiednie wersety z tego samego rozdziału i zwraca rozszerzony kontekst.

Ponieważ wersety zostały zaindeksowane z sekwencyjnymi identyfikatorami całkowitymi (0, 1, 2, …), sąsiedni werset poprzedzający ma `id - 1`, a następny `id + 1`. Użyj metody `client.retrieve()` do pobrania tych punktów bezpośrednio po ID.

Uwaga: id±1 może należeć do innego rozdziału lub nawet innej księgi (na granicy rozdziałów). Po pobraniu punktów odfiltruj te, których numer rozdziału (`metadata.chapter`) różni się od rozdziału wersetu wejściowego.

Funkcja powinna zwracać słownik będący połączeniem oryginalnego wersetu (słownika) oraz:
- `context_text` — połączony tekst sąsiednich wersetów (poprzedni + oryginalny + następny, o ile są w tym samym rozdziale)
- `context_verse_start` — numer pierwszego wersetu w oknie kontekstu
- `context_verse_stop` — numer ostatniego wersetu w oknie kontekstu

Przykłady:
- `Rdz 2,4` → kontekst obejmuje `Rdz 2, 3-5`
- `Rdz 2,1` → kontekst obejmuje `Rdz 2, 1-2` (brak poprzedniego wersetu w tym rozdziale)


In [53]:
def expand_verse_context(verse: dict) -> dict:
    """Fetch the previous and next verse from the same chapter and return the combined context."""
    verse_id = verse["_id"]
    chapter = verse["chapter"]
    book_name = verse["book_name"]

    candidate_ids = [id_ for id_ in [verse_id - 1, verse_id, verse_id + 1] if id_ >= 0]

    context_points = client.retrieve(
        collection_name=COLLECTION_NAME,
        ids=candidate_ids,
        with_payload=True,
        with_vectors=False,
    )

    # Keep only verses from the same chapter (guards against id±1 crossing chapter boundaries)
    same_chapter = sorted(
        [p for p in context_points if p.payload["metadata"]["chapter"] == chapter and p.payload["metadata"]["book_name"] == book_name],
        key=lambda p: p.payload["metadata"]["verse"],
    )

    verse_numbers = [p.payload["metadata"]["verse"] for p in same_chapter]
    combined_text = " ".join(p.payload["page_content"] for p in same_chapter)

    return {
        **verse,
        "context_text": combined_text,
        "context_verse_start": min(verse_numbers),
        "context_verse_stop": max(verse_numbers),
    }

In [54]:
# Przykład 1: werset ze środka rozdziału – powinien zwrócić werset poprzedni i następny
result = rag_query_with_filters(
    "Stworzenie nieba i ziemi", top_k=3, retrieval_mode="hybrid", generate_answer=False
)
result

{'answer': None,
 'context': [{'verse_text': 'Tak macie mówić o nich: "Bogowie, którzy nie stworzyli nieba i ziemi, znikną z ziemi i spod nieba".',
   'bible_part': 'Stary Testament',
   'book_category': 'Księgi prorockie',
   'book_name': 'Księga Jeremiasza',
   'book_abbrev': 'jr',
   'deuterocanonical': False,
   'original_language': 'hebrajski',
   'chapter': 10,
   'verse': 11,
   '_id': 19208,
   '_collection_name': 'biblia'},
  {'verse_text': 'Takie są dzieje stworzenia nieba i ziemi.',
   'bible_part': 'Stary Testament',
   'book_category': 'Pięcioksiąg',
   'book_name': 'Księga Rodzaju',
   'book_abbrev': 'rdz',
   'deuterocanonical': False,
   'original_language': 'hebrajski',
   'chapter': 2,
   'verse': 4,
   '_id': 34,
   '_collection_name': 'biblia'},
  {'verse_text': 'a jeńców uprowadzę na krańce ziemi.  ',
   'bible_part': 'Stary Testament',
   'book_category': 'Księgi historyczne',
   'book_name': 'Księga Judyty',
   'book_abbrev': 'jdt',
   'deuterocanonical': True,
 

In [55]:
verse = result["context"][0]
expanded = expand_verse_context(verse)

print(f"Oryginalny werset:  {expanded['book_name']} {expanded['chapter']},{expanded['verse']}")
print(f"Kontekst (wersety): {expanded['book_name']} {expanded['chapter']}, {expanded['context_verse_start']}-{expanded['context_verse_stop']}")
print(f"Tekst kontekstu:    {expanded['context_text']}")

Oryginalny werset:  Księga Jeremiasza 10,11
Kontekst (wersety): Księga Jeremiasza 10, 10-12
Tekst kontekstu:    Lecz tylko PAN jest prawdziwym Bogiem. On jest Bogiem życia i Królem na wieki. Od Jego gniewu drży ziemia, a gróźb Jego nie mogą znieść narody. Tak macie mówić o nich: "Bogowie, którzy nie stworzyli nieba i ziemi, znikną z ziemi i spod nieba". Ten, który stworzył ziemię swą mocą, utwierdził krąg ziemi swoją mądrością i swym rozumem rozpostarł niebiosa.


In [57]:
# Przykład 2: werset z początku rozdziału – brak poprzedniego wersetu w tym samym rozdziale
# Rdz 1,1 to punkt o id=0 i zarazem pierwszy werset pierwszego rozdziału
first_verse = {
    "_id": 0,
    "book_name": "Księga Rodzaju",
    "chapter": 1,
    "verse": 1,
    "verse_text": "Na początku Bóg stworzył niebo i ziemię.",
}

expanded = expand_verse_context(first_verse)

print(f"Oryginalny werset:  {expanded['book_name']} {expanded['chapter']},{expanded['verse']}")
print(f"Kontekst (wersety): {expanded['book_name']} {expanded['chapter']}, {expanded['context_verse_start']}-{expanded['context_verse_stop']}")
print(f"Tekst kontekstu:    {expanded['context_text']}")

Oryginalny werset:  Księga Rodzaju 1,1
Kontekst (wersety): Księga Rodzaju 1, 1-2
Tekst kontekstu:    Na początku Bóg stworzył niebo i ziemię. A ziemia była bezładną pustką. Ciemność zalegała nad bezmiarem wód, a duch Boży unosił się nad wodami.


In [58]:
# Przykład 3: rozszerzenie kontekstu dla wszystkich wersetów z zapytania
result = rag_query_with_filters(
    "Ile dni trwał biblijny potop?", top_k=4, retrieval_mode="hybrid", generate_answer=False
)

print("Rozszerzone konteksty wersetów:")
for verse in result["context"]:
    expanded = expand_verse_context(verse)
    book = expanded["book_name"]
    ch = expanded["chapter"]
    start = expanded["context_verse_start"]
    stop = expanded["context_verse_stop"]
    label = f"{ch},{start}" if start == stop else f"{ch},{start}-{stop}"
    print(f"\n  {book} {label}")
    print(f"  {expanded['context_text']}")

Rozszerzone konteksty wersetów:

  Księga Rodzaju 7,16-18
  Z wszystkich istot żywych weszły samiec i samica, jak Bóg polecił. Wtedy PAN zamknął za nim drzwi. Potop na ziemi trwał czterdzieści dni. Woda wzbierała i unosiła arkę nad ziemią. Woda wzbierała coraz wyżej nad ziemią, ale arka pływała na wodzie.

  Księga Psalmów 119,83-85
  Chociaż jestem jak skórzany worek w dymie, ustaw Twoich nie zapominam. Ile dni zostało Twemu słudze? Kiedy osądzisz moich prześladowców? Pyszni wykopali dół pode mną, pogardzając Prawem Twoim!

  Ewangelia Łukasza 17,25-27
  Jednak przedtem musi On wiele wycierpieć i być odrzucony przez to pokolenie. I jak było za dni Noego, tak będzie w dniach Syna Człowieczego. Jedli, pili, żenili się, wychodziły za mąż, aż do dnia, w którym Noe wszedł do arki. Potem przyszedł potop i wytracił wszystkich.

  Księga Rodzaju 8,21-22
  Gdy PAN poczuł miłą woń, postanowił: "Nie będę już więcej złorzeczył ziemi z powodu człowieka, gdyż skłonności człowieka są złe od młodości

### Zadanie 7 — Filtrowanie nietrafionych wersetów przez LLM

Napisz funkcję `filter_irrelevant_verses`, która dla listy wersetów (z rozszerzonym kontekstem z ćwiczenia 6) oceni przy użyciu LLM, czy każdy z nich rzeczywiście odpowiada na pytanie użytkownika.

Funkcja powinna przyjmować:
- `user_query` — oryginalne pytanie użytkownika
- `verses` — lista słowników z rozszerzonym kontekstem (pole `context_text` z ćwiczenia 6)

Dla każdego wersetu wyślij do LLM pytanie użytkownika oraz `context_text` (a nie sam werset) — szersze okno kontekstu pozwala LLM lepiej ocenić trafność. Użyj structured output z modelem Pydantic zawierającym pole `is_relevant: bool` oraz uzasadnienie decyzji.

Ponieważ wersetów może być wiele, wywołaj LLM dla wszystkich naraz przy użyciu `chain.batch()` z parametrem `max_concurrency` — to znacznie przyspiesza przetwarzanie w porównaniu do sekwencyjnych wywołań.

Funkcja powinna zwracać słownik z kluczami:
- `relevant_verses` — lista trafnych wersetów, każdy wzbogacony o pole z uzasadnieniem trafności
- `irrelevant_verses` — lista odfiltrowanych wersetów, każdy wzbogacony o pole z uzasadnieniem odrzucenia


In [60]:
class VerseRelevance(BaseModel):
    is_relevant: bool = Field(
        description=(
            "True jeśli fragment Biblii bezpośrednio lub pośrednio dotyczy pytania użytkownika "
            "i może stanowić wartościowy element odpowiedzi na to pytanie."
        )
    )
    reasoning: str = Field(
        description="Krótkie uzasadnienie decyzji o trafności lub braku trafności fragmentu."
    )


SYSTEM_PROMPT_FILTER = (
    "Jesteś ekspertem biblistyki analizującym trafność wersetów Biblii "
    "w odniesieniu do pytania użytkownika. Twoim zadaniem jest ocena, "
    "czy podany fragment Biblii zawiera informacje, które bezpośrednio "
    "lub pośrednio pozwalają odpowiedzieć na pytanie użytkownika.\n\n"
    "Kryteria oceny:\n"
    "1. Czy fragment porusza tę samą tematykę co pytanie?\n"
    "2. Czy zawiera kluczowe pojęcia lub kontekst istotny dla pytania?\n"
    "3. Nie wymagaj, aby fragment był kompletną odpowiedzią – wystarczy, "
    "że jest wartościowym elementem układanki."
)

HUMAN_PROMPT_FILTER = (
    "PYTANIE UŻYTKOWNIKA: {user_query}\n\n"
    "FRAGMENT BIBLII: {bible_quote}"
)

filter_prompt = ChatPromptTemplate.from_messages([
    SystemMessage(SYSTEM_PROMPT_FILTER),
    HumanMessagePromptTemplate.from_template(HUMAN_PROMPT_FILTER),
])

filter_chain = filter_prompt | llm.with_structured_output(
    VerseRelevance  #, method="function_calling"
).with_retry(stop_after_attempt=5, wait_exponential_jitter=True)


def filter_irrelevant_verses(user_query: str, verses: list[dict]) -> dict:
    """Evaluate each expanded verse for relevance to the user query using parallel LLM calls."""
    inputs = [
        {"user_query": user_query, "bible_quote": v["context_text"]}
        for v in verses
    ]

    responses = filter_chain.batch(inputs, config={"max_concurrency": 24})

    relevant_verses, irrelevant_verses = [], []
    for verse, resp in zip(verses, responses):
        if resp.is_relevant:
            relevant_verses.append({**verse, "why_relevant": resp.reasoning})
        else:
            irrelevant_verses.append({**verse, "why_irrelevant": resp.reasoning})

    print(f"Trafne: {len(relevant_verses)} | Odfiltrowane: {len(irrelevant_verses)}")

    return {
        "relevant_verses": relevant_verses,
        "irrelevant_verses": irrelevant_verses,
    }

In [61]:
# Przykład 1: retrieval → expand → filter
# Pytanie o potop – spodziewamy się, że część wersetów (np. o dniach/liczbach niezwiązanych z potopem) zostanie odfiltrowana

query = "Ile dni trwał biblijny potop?"

retrieved = rag_query_with_filters(query, top_k=6, retrieval_mode="hybrid", generate_answer=False)
expanded = [expand_verse_context(v) for v in retrieved["context"]]
filtered = filter_irrelevant_verses(query, expanded)

print("\n--- TRAFNE ---")
for v in filtered["relevant_verses"]:
    label = f"{v['chapter']},{v['context_verse_start']}" if v['context_verse_start'] == v['context_verse_stop'] \
        else f"{v['chapter']},{v['context_verse_start']}-{v['context_verse_stop']}"
    print(f"  {v['book_name']} {label}")
    print(f"  Uzasadnienie: {v['why_relevant']}")

print("\n--- ODFILTROWANE ---")
for v in filtered["irrelevant_verses"]:
    label = f"{v['chapter']},{v['context_verse_start']}" if v['context_verse_start'] == v['context_verse_stop'] \
        else f"{v['chapter']},{v['context_verse_start']}-{v['context_verse_stop']}"
    print(f"  {v['book_name']} {label}")
    print(f"  Uzasadnienie: {v['why_irrelevant']}")

Trafne: 1 | Odfiltrowane: 5

--- TRAFNE ---
  Księga Rodzaju 7,16-18
  Uzasadnienie: Fragment bezpośrednio odnosi się do pytania użytkownika, podając informację, że potop trwał czterdzieści dni. Jest to kluczowa informacja potrzebna do odpowiedzi na pytanie.

--- ODFILTROWANE ---
  Księga Psalmów 119,83-85
  Uzasadnienie: Fragment nie porusza tematyki potopu ani nie zawiera kluczowych pojęć związanych z czasem trwania potopu. Dotyczy raczej osobistych przeżyć i modlitwy o sprawiedliwość.
  Ewangelia Łukasza 17,25-27
  Uzasadnienie: Fragment odnosi się do czasów Noego i wspomina o potopie, ale nie zawiera informacji o tym, ile dni trwał potop. Skupia się na porównaniu dni Noego z dniami Syna Człowieczego, a nie na szczegółach dotyczących długości trwania potopu.
  Księga Ezechiela 4,3-5
  Uzasadnienie: Fragment dotyczy symbolicznego aktu proroka Ezechiela, który ma ponosić karę za winy ludu izraelskiego przez określoną liczbę dni. Nie ma on związku z biblijnym potopem ani nie zawiera in

In [62]:
# Przykład 2: bardziej ogólne pytanie → więcej wersetów do oceny, oczekiwany wyższy odsetek trafnych

query = "Co Biblia mówi o modlitwie?"

retrieved = rag_query_with_filters(query, top_k=8, retrieval_mode="hybrid", generate_answer=False)
expanded = [expand_verse_context(v) for v in retrieved["context"]]
filtered = filter_irrelevant_verses(query, expanded)

print(f"Pobrano: {len(expanded)} | Trafne: {len(filtered['relevant_verses'])} | Odfiltrowane: {len(filtered['irrelevant_verses'])}")

print("\n--- TRAFNE ---")
for v in filtered["relevant_verses"]:
    print(f"  {v['book_name']} {v['chapter']},{v['verse']}: {v['verse_text'][:80]}...")

print("\n--- ODFILTROWANE ---")
for v in filtered["irrelevant_verses"]:
    print(f"  {v['book_name']} {v['chapter']},{v['verse']}: {v['verse_text'][:80]}...")
    print(f"  Powód: {v['why_irrelevant']}")

Trafne: 4 | Odfiltrowane: 4
Pobrano: 8 | Trafne: 4 | Odfiltrowane: 4

--- TRAFNE ---
  Księga Izajasza 37,15: Ezechiasz tak się wtedy modlił do PANA:...
  1 List do Tymoteusza 4,5: Uświęca to przecież słowo Boże i modlitwa....
  Księga Habakuka 3,1: Modlitwa proroka Habakuka na wzór lamentacji....
  Księga Hioba 21,15: Kim jest Wszechmocny, abyśmy Mu służyli, jaką korzyść przynosi modlitwa do Niego...

--- ODFILTROWANE ---
  Księga Izajasza 45,10: Biada temu, kto mówi do ojca: "Co ty spłodziłeś?", a do kobiety: "Co ty urodziła...
  Powód: Fragment dotyczy relacji między stworzeniem a Stwórcą, podkreślając ograniczenia człowieka w kwestionowaniu Bożych działań. Nie odnosi się bezpośrednio do modlitwy ani nie zawiera kluczowych pojęć związanych z modlitwą.
  Księga Jeremiasza 45,4: Tak mu powiesz: Tak mówi PAN: W całym kraju burzę, co zbudowałem, i wyrywam, co ...
  Powód: Fragment dotyczy wyroczni PANA i nieszczęścia sprowadzanego na ludzi, a nie modlitwy. Nie zawiera kluczowych pojęć a

### Zadanie 8 — Grupowanie wersetów w kategorie tematyczne

Napisz funkcję `group_verses`, która pogrupuje trafne wersety w kategorie tematyczne odpowiadające różnym aspektom odpowiedzi na pytanie użytkownika.

Funkcja przyjmuje:
- `user_query` — oryginalne pytanie użytkownika
- `relevant_verses` — lista trafnych wersetów z zadania 7

Przygotuj dla LLM wejście w postaci listy słowników `{"id": ..., "text": ...}` (gdzie `text` to `context_text`) i przekaż je jako wiadomość ludzką w JSON. W wiadomości systemowej poinstruuj LLM, żeby podzielił wszystkie cytaty na niezależne tematycznie grupy i nazwał je tak, by nadawały się jako nagłówki sekcji odpowiedzi. Każdy werset musi trafić do przynajmniej jednej kategorii; jeśli pasuje do kilku — może pojawić się w wielu.

Użyj structured output — zdefiniuj modele Pydantic. Następnie za pomocą słownika `{id: verse}` rozwiąż ID z powrotem na pełne słowniki wersetów.

Funkcja powinna zwracać słownik `{nazwa_kategorii: [verse_dict, ...]}`.

Aby zobaczyć jak powinien wyglądać efekt grupowania w praktyce, odwiedź stronę [biblia.palej.dev](https://biblia.palej.dev/) i przejrzyj przykładowe analizy — nazwy aspektów to nagłówki poszczególnych sekcji odpowiedzi.


In [64]:
filtered

{'relevant_verses': [{'verse_text': 'Ezechiasz tak się wtedy modlił do PANA:',
   'bible_part': 'Stary Testament',
   'book_category': 'Księgi prorockie',
   'book_name': 'Księga Izajasza',
   'book_abbrev': 'iz',
   'deuterocanonical': False,
   'original_language': 'hebrajski',
   'chapter': 37,
   'verse': 15,
   '_id': 18363,
   '_collection_name': 'biblia',
   'context_text': 'Ezechiasz odebrał list od posłów i przeczytał go. Potem poszedł do świątyni PANA i rozwinął go przed PANEM. Ezechiasz tak się wtedy modlił do PANA: "PANIE Zastępów, Boże Izraela, który zasiadasz na cherubach! Tylko Ty jeden jesteś Bogiem wszystkich królestw ziemi! Ty uczyniłeś niebo i ziemię!',
   'context_verse_start': 14,
   'context_verse_stop': 16,
   'why_relevant': 'Fragment dotyczy modlitwy, ponieważ opisuje, jak Ezechiasz modli się do Boga w świątyni. Zawiera kluczowe pojęcia związane z modlitwą, takie jak zwracanie się do Boga, uznanie Jego wszechmocy i obecności. Jest to przykład modlitwy w Biblii,

In [69]:
import json
from langchain_core.prompts import SystemMessagePromptTemplate
from langchain.messages import HumanMessage


class CategoryEntry(BaseModel):
    category_name: str = Field(description="Nazwa obszaru tematycznego – pasuje jako nagłówek sekcji odpowiedzi.")
    verse_ids: list[int] = Field(description="Lista id wersetów należących do tego obszaru tematycznego.")


class GroupedVerses(BaseModel):
    categories: list[CategoryEntry] = Field(
        description=(
            "Lista obszarów tematycznych. Obszary muszą być odrębne i niezależne tematycznie. "
            "Każde id wersetu musi wystąpić w co najmniej jednej kategorii. "
            "Jeśli werset pasuje do kilku tematów, możesz umieścić go w wielu kategoriach."
        )
    )


_SYSTEM_PROMPT_GROUP_VERSES = (
    "Jesteś specjalistą od analizy Biblii. Twoim zadaniem jest "
    "pogrupowanie cytatów Biblii w kategorie tematyczne. Dostaniesz "
    "obiekt JSON, który jest listą słowników posiadających id oraz tekst "
    "cytatu z Biblii. Podziel wszystkie cytaty na niezależne i odrębne "
    "tematycznie grupy. Do każdej grupy przypisz jedno lub więcej id "
    "cytatów, które będą należały do tej grupy. Musisz wykorzystać id "
    "wszystkich dostarczonych cytatów co najmniej raz. Jeśli któryś "
    "fragment pasuje do wielu tematów, możesz go umieścić w kilku "
    "grupach.\n\n"
    "Te grupy będą stanowiły plan ramowy paragrafów tekstu, w którym "
    "znajdzie się odpowiedź na zadane przez użytkownika pytanie dotyczące "
    "Biblii. Nazwij więc grupy tak, żeby pasowały na nagłówki tekstu "
    "z odpowiedzią dla użytkownika i postaraj się, żeby liczba sekcji nie "
    "była ani zbyt duża ani zbyt mała.\n\n"
    "Pytanie, które zadał użytkownik: {user_query}"
)


def group_verses(user_query: str, relevant_verses: list[dict]) -> dict:
    """Group relevant verses into thematic aspects using LLM structured output."""
    grouping_input = [
        {"id": v["_id"], "text": v["context_text"]}
        for v in relevant_verses
    ]

    prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(_SYSTEM_PROMPT_GROUP_VERSES),
        HumanMessage(json.dumps(grouping_input, ensure_ascii=False)),
    ])

    chain = prompt | llm.with_structured_output(GroupedVerses, method="function_calling")
    result = chain.invoke({"user_query": user_query})

    verse_lookup = {v["_id"]: v for v in relevant_verses}
    grouped = {
        entry.category_name: [
            verse_lookup[v_id] for v_id in entry.verse_ids if v_id in verse_lookup
        ]
        for entry in result.categories
    }

    print(f"Pogrupowano {len(relevant_verses)} wersetów → {len(grouped)} kategorii: {list(grouped.keys())}")
    return grouped

In [70]:
# Przykład 1: pełny pipeline retrieval → expand → filter → group
query = "Co Biblia mówi o miłości bliźniego?"

retrieved = rag_query_with_filters(query, top_k=10, retrieval_mode="hybrid", generate_answer=False)
expanded = [expand_verse_context(v) for v in retrieved["context"]]
filtered = filter_irrelevant_verses(query, expanded)
grouped = group_verses(query, filtered["relevant_verses"])

print()
for category, verses in grouped.items():
    print(f"### {category} ({len(verses)} wersetów)")
    for v in verses:
        print(f"  {v['book_name']} {v['chapter']},{v['verse']}: {v['verse_text'][:80]}")
    print()

Trafne: 5 | Odfiltrowane: 5
Pogrupowano 5 wersetów → 4 kategorii: ['Miłość bliźniego jako przykazanie', 'Miłość bliźniego a miłość do Boga', 'Miłość bliźniego w praktyce', 'Miłość jako pełnia Prawa']

### Miłość bliźniego jako przykazanie (2 wersetów)
  List do Rzymian 13,10: Miłość nie wyrządza zła bliźniemu. Miłość jest więc pełnią Prawa.
  Ewangelia Marka 12,31: Drugie jest to: Będziesz miłował swego bliźniego jak siebie samego. Nie ma inneg

### Miłość bliźniego a miłość do Boga (2 wersetów)
  1 List Jana 4,21: Takie otrzymaliśmy od Niego przykazanie, aby ten, kto miłuje Boga, miłował też s
  Ewangelia Marka 12,31: Drugie jest to: Będziesz miłował swego bliźniego jak siebie samego. Nie ma inneg

### Miłość bliźniego w praktyce (2 wersetów)
  Mądrość Syracha 29,1: Kto jest miłosierny pożycza bliźniemu kto go wspomaga swą ręką zachowuje przykaz
  Ewangelia Mateusza 5,46: Jeśli bowiem miłujecie tylko tych, którzy was miłują, jakiej zapłaty możecie się

### Miłość jako pełnia Prawa (1 

In [71]:
# Przykład 2: pytanie wieloaspektowe – spodziewamy się kilku kategorii
query = "Czego Biblia uczy o pieniądzach i bogactwie?"

retrieved = rag_query_with_filters(query, top_k=12, retrieval_mode="dense", generate_answer=False)
expanded = [expand_verse_context(v) for v in retrieved["context"]]
filtered = filter_irrelevant_verses(query, expanded)
grouped = group_verses(query, filtered["relevant_verses"])

print()
for category, verses in grouped.items():
    print(f"### {category}")
    for v in verses:
        print(f"  {v['book_name']} {v['chapter']},{v['verse']}: {v['verse_text'][:80]}")
    print()

Trafne: 7 | Odfiltrowane: 5
Pogrupowano 7 wersetów → 3 kategorii: ['Mądrość i bogactwo', 'Bogactwo a moralność', 'Bogactwo i status społeczny']

### Mądrość i bogactwo
  Księga Mądrości 8,5: Jeśli bogactwo jest w życiu czymś upragnionym kto jest bogatszy od mądrości któr
  Mądrość Syracha 10,31: Kogo szanują w biedzie tym bardziej uczczą go w bogactwie kim gardzą w bogactwie
  Mądrość Syracha 10,22: Czy to bogaty godzien sławy czy biedny jego dumą jest bojaźń Pana.  

### Bogactwo a moralność
  Mądrość Syracha 11,18: Kto bogaci się na skąpstwie i chciwości taką będzie miał nagrodę: 
  Księga Przysłów 28,8: Kto powiększa bogactwo lichwą i odsetkami, gromadzi je dla tego, który lituje si
  Mądrość Syracha 13,24: Dobre jest bogactwo w którym nie ma grzechu a ubóstwo jest złem w ustach bezbożn
  Księga Przysłów 15,6: W domu prawego jest wiele bogactw, a w dobytku bezbożnika panuje nieład.

### Bogactwo i status społeczny
  Mądrość Syracha 13,24: Dobre jest bogactwo w którym nie ma grzechu 

### Zadanie 9 — Pobieranie pełnych tekstów rozdziałów

Napisz funkcję `retrieve_chapters`, która dla każdej kategorii z wyników zadania 8 pobierze pełne teksty rozdziałów, do których należą pogrupowane wersety.

Funkcja przyjmuje słownik `grouped_verses` (wynik `group_verses`) i zwraca słownik o tej samej strukturze kluczy, ale zamiast wersetów — listę rozdziałów: `{nazwa_kategorii: [chapter_dict, ...]}`.

Każdy `chapter_dict` powinien zawierać: identyfikator rozdziału (np. `"Księga Rodzaju 1"`), nazwę księgi, numer rozdziału oraz pełny tekst rozdziału złożony ze wszystkich jego wersetów.

Wskazówki implementacyjne:
- Do pobrania wszystkich wersetów danego rozdziału użyj `client.scroll()` z filtrem na `book_name` i `chapter` — zwróci wszystkie punkty spełniające warunek bez limitu wektorowego.
- Złóż tekst rozdziału łącząc kolejne wersety w formacie `numer_wersetu: treść_wersetu` oddzielone znakiem nowej linii.
- Zaimplementuj cache słownikowy indeksowany kluczem `"Nazwa księgi numer_rozdziału"` — ten sam rozdział może być potrzebny dla kilku kategorii, a cache zapobiegnie wielokrotnym zapytaniom do bazy.


In [73]:
def retrieve_chapters(grouped_verses: dict) -> dict:
    """For each category, fetch the full text of every chapter referenced by its verses."""
    grouped_chapters = {}
    chapter_cache = {}

    for category, verses in grouped_verses.items():
        unique_chapters = list({(v["book_name"], v["chapter"]) for v in verses})
        category_chapters = []

        for book_name, chapter_no in unique_chapters:
            cache_key = f"{book_name} {chapter_no}"

            if cache_key not in chapter_cache:
                chapter_points, _ = client.scroll(
                    collection_name=COLLECTION_NAME,
                    scroll_filter=models.Filter(
                        must=[
                            models.FieldCondition(
                                key="metadata.book_name",
                                match=qdrant_models.MatchValue(value=book_name),
                            ),
                            models.FieldCondition(
                                key="metadata.chapter",
                                match=qdrant_models.MatchValue(value=chapter_no),
                            ),
                        ]
                    ),
                    with_payload=True,
                    with_vectors=False,
                    limit=1000,
                )
                chapter_text = "\n".join(
                    f"{p.payload['metadata']['verse']}: {p.payload['page_content']}"
                    for p in chapter_points
                )
                chapter_cache[cache_key] = chapter_text
            else:
                chapter_text = chapter_cache[cache_key]

            category_chapters.append({
                "chapter_id": cache_key,
                "book_name": book_name,
                "chapter_number": chapter_no,
                "chapter_text": chapter_text,
            })

        grouped_chapters[category] = category_chapters

    n_unique = len(chapter_cache)
    print(f"Pobrano {n_unique} unikalnych rozdziałów dla {len(grouped_chapters)} kategorii")
    return grouped_chapters

In [74]:
# Przykład 1: kontynuacja pipeline z ćwiczenia 8 – pobieramy rozdziały dla pogrupowanych wersetów
# Używamy `grouped` z poprzedniego przykładu (query: "Co Biblia mówi o miłości bliźniego?")

chapters = retrieve_chapters(grouped)

print()
for category, chaps in chapters.items():
    print(f"=== {category} ===")
    for ch in chaps:
        lines = ch["chapter_text"].split("\n")
        print(f"  {ch['book_name']} rozdz. {ch['chapter_number']} ({len(lines)} wersetów)")
        # Pokaż pierwsze 3 wiersze rozdziału
        for line in lines[:3]:
            print(f"    {line}")
        if len(lines) > 3:
            print(f"    ... (+{len(lines) - 3} wersetów)")
    print()

Pobrano 6 unikalnych rozdziałów dla 3 kategorii

=== Mądrość i bogactwo ===
  Księga Mądrości rozdz. 8 (21 wersetów)
    1: Jej moc sięga krańców świata i wszystkim zarządza wspaniale. 
    2: Zakochałem się w niej za młodu zapragnąłem jej ponad wszystko szukałem sposobu aby ją poślubić gdyż zachwyciłem się jej pięknem.  
    3: Ona się chlubi szlachetnym pochodzeniem i ma udział w życiu Boga a Władca wszechświata otacza ją miłością.  
    ... (+18 wersetów)
  Mądrość Syracha rozdz. 10 (31 wersetów)
    1: Mądry sędzia wychowuje swój naród a roztropne rządy wprowadzają ład.  
    2: Jaki przywódca ludu tacy i jego słudzy mieszkańcy są podobni do tego kto nimi rządzi.  
    3: Król bez zasad rujnuje swój naród mądrość władców wpływa na rozwój miasta.  
    ... (+28 wersetów)

=== Bogactwo a moralność ===
  Mądrość Syracha rozdz. 13 (26 wersetów)
    1: Ubrudzi się ten kto dotyka smoły kto trzyma z pyszałkiem upodabnia się do niego.  
    2: Nie podnoś ciężaru ponad siły nie zadawaj się 

In [75]:
# Przykład 2: sprawdzamy cache – te same rozdziały nie są pobierane ponownie
# Nowe zapytanie, kilka wersetów ze znanych rozdziałów

query2 = "Czego Biblia uczy o pieniądzach i bogactwie?"
retrieved2 = rag_query_with_filters(query2, top_k=10, retrieval_mode="hybrid", generate_answer=False)
expanded2 = [expand_verse_context(v) for v in retrieved2["context"]]
filtered2 = filter_irrelevant_verses(query2, expanded2)
grouped2 = group_verses(query2, filtered2["relevant_verses"])
chapters2 = retrieve_chapters(grouped2)

print()
for category, chaps in chapters2.items():
    print(f"=== {category} ===")
    for ch in chaps:
        print(f"  {ch['book_name']} rozdz. {ch['chapter_number']}")

Trafne: 3 | Odfiltrowane: 7
Pogrupowano 3 wersetów → 3 kategorii: ['Mądrość jako prawdziwe bogactwo', 'Odpowiedzialność za powierzone dobra', 'Szacunek i status społeczny a bogactwo']
Pobrano 3 unikalnych rozdziałów dla 3 kategorii

=== Mądrość jako prawdziwe bogactwo ===
  Księga Mądrości rozdz. 8
=== Odpowiedzialność za powierzone dobra ===
  Ewangelia Łukasza rozdz. 19
=== Szacunek i status społeczny a bogactwo ===
  Mądrość Syracha rozdz. 10


### Zadanie 10 — Generowanie odpowiedzi per aspekt

Napisz funkcję `generate_answers_per_aspect`, która wygeneruje ustrukturyzowaną odpowiedź dla każdej kategorii tematycznej z ćwiczenia 8.

Funkcja przyjmuje:
- `user_query` — oryginalne pytanie użytkownika
- `grouped_verses` — wynik `group_verses` (słownik kategorii z wersetami)
- `retrieved_chapters` — wynik `retrieve_chapters` (słownik kategorii z rozdziałami)

Dla każdego aspektu/kategorii zbuduj kontekst łączący dwa poziomy szczegółowości: najpierw wylistuj konkretne trafne wersety z danego rozdziału (numer i treść), a potem dołącz pełny tekst rozdziału — dzięki temu LLM ma zarówno celne fragmenty, jak i szerszy kontekst narracyjny.

Użyj structured output z hierarchicznym modelem Pydantic:
- `BibleCitation` — nazwa księgi, numer rozdziału, numer wersetu
- `ResponseParagraph` — tekst akapitu + lista cytowań (`list[BibleCitation]`)
- `AspectResponse` — nazwa aspektu + lista akapitów (`list[ResponseParagraph]`)

Wszystkie aspekty przetwarzaj równolegle przy użyciu `chain.batch()` z `max_concurrency`.

Funkcja powinna zwracać listę słowników (`.model_dump()` na każdym `AspectResponse`) — jednego na każdą kategorię.


In [76]:
from textwrap import dedent as _dedent


class _BibleCitation(BaseModel):
    book_name: str = Field(
        description="Pełna nazwa księgi Biblii – użyj dokładnie takiej nazwy jak w kontekście."
    )
    chapter_number: int = Field(description="Numer rozdziału cytowanego fragmentu.")
    verse_number: int = Field(description="Numer wersetu cytowanego fragmentu.")


class _ResponseParagraph(BaseModel):
    response_paragraph_text: str = Field(
        description="Tekst akapitu odpowiedzi dotyczący jednego konkretnego zagadnienia."
    )
    response_paragraph_citations: list[_BibleCitation] = Field(
        description=(
            "Lista cytowań biblijnych użytych w tym akapicie. "
            "Cytuj wyłącznie wersety wprost wymienione w kontekście."
        )
    )


class _AspectResponse(BaseModel):
    aspect_name: str = Field(description="Nazwa aspektu, którego dotyczy odpowiedź.")
    paragraphs: list[_ResponseParagraph] = Field(
        description=(
            "Lista akapitów poruszających różne zagadnienia z kontekstu. "
            "Razem powinny tworzyć spójną, ciągłą narrację w ramach danego aspektu."
        )
    )


_SYSTEM_PROMPT_GENERATE = _dedent("""
    Jesteś asystentem analizy Biblii, który pomaga użytkownikowi
    poznać stanowisko Biblii w określonych kwestiach. Dostaniesz
    w kolejnej wiadomości: oryginalne pytanie użytkownika, wersety,
    które dotyczą tego pytania, a także całe rozdziały, w których
    znajdują się te wersety dla lepszego kontekstu. Zostaniesz
    również poinformowany jakiego aspektu dotyczą te wersety,
    ponieważ zostały one przypisane do konkretnego aspektu.
    Twoim zadaniem jest odpowiedzieć na pytanie użytkownika
    w kontekście przytoczonych fragmentów Biblii w odniesieniu
    do wskazanego aspektu.

    Zasada krytyczna: odpowiadaj wyłącznie w oparciu o przytoczony
    kontekst. Nie używaj własnej wiedzy!!! Nie dokonuj własnej
    interpretacji!!! POD ŻADNYM POZOREM nie używaj wiedzy
    spoza dostarczonego tekstu.

    Twoja odpowiedź powinna być zwięzła i składać się z krótkich,
    trafnych akapitów przekazujących konkretną myśl. Każdy akapit
    powinien obejmować listę cytowań wersetów, które popierają tę myśl.
    Cała odpowiedź powinna być spójna i prowadzić jedną ciągłą narrację.
    Cytowania mogą pochodzić WYŁĄCZNIE z przytoczonych wersetów.

    Cytowania (sigla) podawaj w zapisie katolickim, np. Rdz 1, 2 zamiast Rdz 1:2.

    Jeśli powołujesz się na dokładny cytat tekstu Biblii, zamknij go
    w backtikach oraz cudzysłowiach, np. `"Ten tekst jest fragmentem Biblii"`.
    Nie używaj dla cytatów pogrubienia ani znaków *.
    Cudzysłowy powinny być dokładnie tym znakiem -> " zarówno otwierający jak i zamykający.
""")

_HUMAN_PROMPT_GENERATE = _dedent("""
    Pytanie użytkownika: {user_query}

    Aspekt, który masz poruszyć: {aspect}

    Kontekst: {context}
""")

_generate_prompt = ChatPromptTemplate.from_messages([
    SystemMessage(_SYSTEM_PROMPT_GENERATE),
    HumanMessagePromptTemplate.from_template(_HUMAN_PROMPT_GENERATE),
])

_generate_chain = _generate_prompt | llm.with_structured_output(
    _AspectResponse, method="function_calling"
)


def _build_bible_context(aspect_verses: list[dict], aspect_chapters: list[dict]) -> str:
    context = ""
    for chapter in sorted(aspect_chapters, key=lambda c: c["book_name"]):
        verses_in_chapter = [
            v for v in aspect_verses
            if v["book_name"] == chapter["book_name"] and v["chapter"] == chapter["chapter_number"]
        ]
        verses_str = "".join(
            f"Numer wersetu: {v['verse']} | Treść wersetu: {v['verse_text']}\n"
            for v in verses_in_chapter
        )
        context += _dedent(f"""
            Księga: {chapter['book_name']} | Numer rozdziału: {chapter['chapter_number']}
            Wersety:
            {verses_str}
            Treść całego rozdziału:
            {chapter['chapter_text']}
            =========================================
        """)
    return context


def generate_answers_per_aspect(
    user_query: str,
    grouped_verses: dict,
    retrieved_chapters: dict,
) -> list[dict]:
    """Generate a structured answer for each thematic aspect using the verse + chapter context."""
    inputs = []
    aspects = list(grouped_verses.keys())

    for aspect in aspects:
        bible_context = _build_bible_context(
            aspect_verses=grouped_verses[aspect],
            aspect_chapters=retrieved_chapters[aspect],
        )
        inputs.append({"user_query": user_query, "aspect": aspect, "context": bible_context})

    responses = _generate_chain.batch(inputs, config={"max_concurrency": 8})

    results = []
    for resp in responses:
        results.append(resp.model_dump())

    print(f"Wygenerowano odpowiedzi dla {len(results)} aspektów")
    return results

In [77]:
# Przykład 1: pełny pipeline end-to-end
query = "Co Biblia mówi o miłości bliźniego?"

retrieved = rag_query_with_filters(query, top_k=10, retrieval_mode="hybrid", generate_answer=False)
expanded  = [expand_verse_context(v) for v in retrieved["context"]]
filtered  = filter_irrelevant_verses(query, expanded)
grouped   = group_verses(query, filtered["relevant_verses"])
chapters  = retrieve_chapters(grouped)
answers   = generate_answers_per_aspect(query, grouped, chapters)

for aspect_answer in answers:
    print(f"## {aspect_answer['aspect_name']}")
    for para in aspect_answer["paragraphs"]:
        citations = ", ".join(
            f"{c['book_name']} {c['chapter_number']},{c['verse_number']}"
            for c in para["response_paragraph_citations"]
        )
        print(f"  {para['response_paragraph_text']}")
        print(f"  [{citations}]")
    print()

Trafne: 5 | Odfiltrowane: 5
Pogrupowano 5 wersetów → 4 kategorii: ['Miłość bliźniego jako przykazanie', 'Miłość bliźniego a miłość Boga', 'Miłość bliźniego w praktyce', 'Uniwersalność miłości bliźniego']
Pobrano 5 unikalnych rozdziałów dla 4 kategorii
Wygenerowano odpowiedzi dla 4 aspektów
## Miłość bliźniego jako przykazanie
  W Ewangelii Marka miłość bliźniego jest przedstawiona jako jedno z dwóch największych przykazań. Jezus mówi: "Drugie jest to: Będziesz miłował swego bliźniego jak siebie samego. Nie ma innego przykazania większego od tych" (Mk 12, 31). To wskazuje na fundamentalne znaczenie miłości bliźniego w nauczaniu Jezusa, stawiając ją na równi z miłością do Boga.
  [Ewangelia Marka 12,31]
  List do Rzymian podkreśla, że miłość bliźniego jest pełnią Prawa. Apostoł Paweł pisze: "Miłość nie wyrządza zła bliźniemu. Miłość jest więc pełnią Prawa" (Rz 13, 10). Wskazuje to, że wszystkie przykazania można sprowadzić do zasady miłości, która nie krzywdzi innych, a tym samym wypełni

In [91]:
# Przykład 2: surowy wynik – słownik z paragrafami i cytowaniami
import pprint
pprint.pprint(answers[0])

{'aspect_name': 'Miłość bliźniego jako przykazanie',
 'paragraphs': [{'response_paragraph_citations': [{'book_name': 'Ewangelia '
                                                                'Marka',
                                                   'chapter_number': 12,
                                                   'verse_number': 31}],
                 'response_paragraph_text': 'W Ewangelii Marka miłość '
                                            'bliźniego jest przedstawiona jako '
                                            'jedno z dwóch największych '
                                            'przykazań. Jezus mówi: "Drugie '
                                            'jest to: Będziesz miłował swego '
                                            'bliźniego jak siebie samego. Nie '
                                            'ma innego przykazania większego '
                                            'od tych" (Mk 12, 31). To wskazuje '
                           

In [92]:
answers[0]

{'aspect_name': 'Miłość bliźniego jako przykazanie',
 'paragraphs': [{'response_paragraph_text': 'W Ewangelii Marka miłość bliźniego jest przedstawiona jako jedno z dwóch największych przykazań. Jezus mówi: "Drugie jest to: Będziesz miłował swego bliźniego jak siebie samego. Nie ma innego przykazania większego od tych" (Mk 12, 31). To wskazuje na fundamentalne znaczenie miłości bliźniego w nauczaniu Jezusa, stawiając ją na równi z miłością do Boga.',
   'response_paragraph_citations': [{'book_name': 'Ewangelia Marka',
     'chapter_number': 12,
     'verse_number': 31}]},
  {'response_paragraph_text': 'List do Rzymian podkreśla, że miłość bliźniego jest pełnią Prawa. Apostoł Paweł pisze: "Miłość nie wyrządza zła bliźniemu. Miłość jest więc pełnią Prawa" (Rz 13, 10). Wskazuje to, że wszystkie przykazania można sprowadzić do zasady miłości, która nie krzywdzi innych, a tym samym wypełnia Prawo.',
   'response_paragraph_citations': [{'book_name': 'List do Rzymian',
     'chapter_number': 

## Pipeline w langgraph

### Zadanie 11 — Pipeline RAG w LangGraph

Przekształć funkcje z zadań 5–10 w spójny pipeline oparty na LangGraph. Celem jest zamknięcie całego przepływu — od pytania użytkownika aż po listę wygenerowanych odpowiedzi — w jeden graf, który można wywołać jedną instrukcją.

**Struktura grafu**

Graf jest liniowy (brak rozgałęzień warunkowych) i składa się z siedmiu węzłów wykonywanych sekwencyjnie:

```
START
  └─► generate_requests      # zadanie 5: generuje listę zapytań do retrievera
  └─► execute_retrieval      # zadanie 2/4: pobiera wersety z Qdrant
  └─► expand_neighbors       # zadanie 6: rozszerza kontekst o sąsiednie wersety
  └─► filter_irrelevant      # zadanie 7: filtruje nieistotne wersety przez LLM
  └─► group_results          # zadanie 8: grupuje trafne wersety w kategorie
  └─► retrieve_chapters      # zadanie 9: pobiera pełne teksty rozdziałów
  └─► generate_answer        # zadanie 10: generuje odpowiedź per aspekt
END
```

**Stan grafu**

Zdefiniuj klasę stanu jako `TypedDict`. Powinna zawierać:
- `user_query` — wejściowe pytanie użytkownika
- po jednym polu na dane wyjściowe każdego węzła (np. `retrieval_requests`, `retrieved_verses`, `expanded_verses`, `relevant_verses`, `grouped_verses`, `retrieved_chapters`, `answers`)